# Notebook 4 — Model Training

Trains XGBoost, CNN, and probability-averaging ensembles for Signature, GARCH, and Combined feature sets across all selected experiments.

In [8]:
import json
import gc
import sys
import time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from xgboost import XGBClassifier

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except ImportError:
    cp = None
    CUPY_AVAILABLE = False

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

In [9]:
print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nTensorFlow version:")
print(tf.__version__)

print("\nTensorFlow GPUs:")
print(tf.config.list_physical_devices("GPU"))

print("\nCuPy available:")
print(CUPY_AVAILABLE)

if CUPY_AVAILABLE:
    try:
        print("\nCuPy CUDA runtime:")
        print(cp.cuda.runtime.runtimeGetVersion())

        print("\nCuPy GPU:")
        device_id = cp.cuda.Device().id
        properties = cp.cuda.runtime.getDeviceProperties(device_id)
        print(properties["name"].decode())
    except Exception as exc:
        print("CuPy imported but CUDA initialization failed:")
        print(exc)

Python executable:
/home/kfung03/finance-gpu-env/bin/python

Python version:
3.12.13 (main, Jul 23 2026, 14:43:28) [Clang 22.1.3 ]

TensorFlow version:
2.21.0

TensorFlow GPUs:
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

CuPy available:
False


In [10]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
FEATURE_ROOT = Path("../data/features")
MODEL_ROOT = Path("../models")
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

In [11]:
# Must run before building any TensorFlow models.
TF_GPUS = tf.config.list_physical_devices("GPU")

if TF_GPUS:
    try:
        for gpu in TF_GPUS:
            tf.config.experimental.set_memory_growth(gpu, True)

        # Helpful for modern NVIDIA GPUs.
        tf.keras.mixed_precision.set_global_policy("mixed_float16")

        print("TensorFlow GPU detected:", TF_GPUS)
        print(
            "TensorFlow policy:",
            tf.keras.mixed_precision.global_policy(),
        )

    except RuntimeError as exc:
        print("TensorFlow GPU was already initialized:", exc)

else:
    # Do not use float16 on a normal CPU.
    tf.keras.mixed_precision.set_global_policy("float32")
    print("TensorFlow GPU not detected; CNN will use CPU.")

TensorFlow GPU detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow policy: <DTypePolicy "mixed_float16">


In [12]:
RUN_EXPERIMENTS = [
    "monthly_ff3",
    "monthly_ff5",
    "daily_ff3",
    "daily_ff5",
]

EXPERIMENT_CONFIG = {
    "monthly_ff3": {"frequency": "monthly", "factor_model": "ff3", "window": 12, "periods_per_year": 12},
    "monthly_ff5": {"frequency": "monthly", "factor_model": "ff5", "window": 12, "periods_per_year": 12},
    "daily_ff3": {"frequency": "daily", "factor_model": "ff3", "window": 60, "periods_per_year": 252},
    "daily_ff5": {"frequency": "daily", "factor_model": "ff5", "window": 60, "periods_per_year": 252},
}

unknown = set(RUN_EXPERIMENTS) - set(EXPERIMENT_CONFIG)
if unknown:
    raise ValueError(f"Unknown experiments: {sorted(unknown)}")

In [15]:
import gc
import time
import numpy as np
import pandas as pd
import tensorflow as tf

FEATURE_NAMES = ["signature", "garch", "combined"]

# Confirm GPU before starting.
TF_GPUS = tf.config.list_physical_devices("GPU")

if not TF_GPUS:
    raise RuntimeError("TensorFlow cannot detect a GPU in this kernel.")

for gpu in TF_GPUS:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        # GPU may already be initialized.
        pass

tf.keras.mixed_precision.set_global_policy("mixed_float16")

print("TensorFlow GPUs:", TF_GPUS)
print("Precision policy:", tf.keras.mixed_precision.global_policy())


for experiment in RUN_EXPERIMENTS:
    experiment_start = time.perf_counter()

    print()
    print("=" * 80)
    print(f"Experiment: {experiment}")
    print("=" * 80)

    feature_dir = FEATURE_ROOT / experiment
    model_dir = MODEL_ROOT / experiment
    model_dir.mkdir(parents=True, exist_ok=True)

    results_path = model_dir / "final_model_results.csv"

    # Skip experiments that already completed.
    if results_path.exists():
        print(f"Skipping completed experiment: {experiment}")
        continue

    # ------------------------------------------------------------------
    # Load labels
    # ------------------------------------------------------------------
    y_train = np.load(
        feature_dir / "y_train.npy"
    ).astype(np.int32, copy=False)

    y_val = np.load(
        feature_dir / "y_validation.npy"
    ).astype(np.int32, copy=False)

    y_test = np.load(
        feature_dir / "y_test.npy"
    ).astype(np.int32, copy=False)

    positive_count = np.count_nonzero(y_train == 1)
    negative_count = np.count_nonzero(y_train == 0)

    scale_pos_weight = (
        negative_count / max(positive_count, 1)
    )

    class_weight = {
        0: 1.0,
        1: float(scale_pos_weight),
    }

    print(f"Train rows: {len(y_train):,}")
    print(f"Validation rows: {len(y_val):,}")
    print(f"Test rows: {len(y_test):,}")
    print(f"Positive-class weight: {scale_pos_weight:.3f}")

    # ------------------------------------------------------------------
    # Load features
    # ------------------------------------------------------------------
    feature_sets = {}

    for feature in FEATURE_NAMES:
        feature_sets[feature] = {
            split: np.load(
                feature_dir / f"X_{feature}_{split}.npy"
            ).astype(np.float32, copy=False)
            for split in ["train", "validation", "test"]
        }

        print(
            f"{feature.title():10} shape:",
            feature_sets[feature]["train"].shape,
        )

    # ------------------------------------------------------------------
    # Result containers
    # ------------------------------------------------------------------
    xgb_models = {}
    cnn_models = {}

    val_probs = {}
    test_probs = {}
    thresholds = {}
    ensemble_settings = {}
    results = []

    # ------------------------------------------------------------------
    # Train each feature set
    # ------------------------------------------------------------------
    for feature, data in feature_sets.items():
        feature_start = time.perf_counter()

        print()
        print("-" * 80)
        print(f"Training feature set: {feature}")
        print("-" * 80)

        X_train = data["train"]
        X_val = data["validation"]
        X_test = data["test"]

        # ==============================================================
        # XGBoost
        # ==============================================================
        xgb_start = time.perf_counter()

        xgb = train_xgb(
            X_train,
            y_train,
            X_val,
            y_val,
            scale_pos_weight,
            use_cuda=True,
        )

        # NumPy prediction inputs are acceptable here.
        # XGBoost may emit one device-mismatch warning.
        xgb_val = (
            xgb.predict_proba(X_val)[:, 1]
            .astype(np.float32, copy=False)
        )

        xgb_test = (
            xgb.predict_proba(X_test)[:, 1]
            .astype(np.float32, copy=False)
        )

        # This was missing in the previous version.
        xgb_threshold = find_best_threshold(
            y_val,
            xgb_val,
        )

        results.append(
            evaluate(
                y_test,
                xgb_test,
                xgb_threshold,
                f"XGBoost {feature.title()} — Test",
            )
        )

        xgb.save_model(
            model_dir / f"xgb_{feature}.json"
        )

        xgb_models[feature] = xgb

        val_probs[("xgb", feature)] = xgb_val
        test_probs[("xgb", feature)] = xgb_test
        thresholds[("xgb", feature)] = xgb_threshold

        print(
            f"XGBoost completed in "
            f"{time.perf_counter() - xgb_start:.1f} seconds"
        )
        print(f"XGBoost threshold: {xgb_threshold:.2f}")

        # ==============================================================
        # CNN
        # ==============================================================
        cnn_start = time.perf_counter()

        # Release the previous Keras graph before creating a new model.
        tf.keras.backend.clear_session()
        gc.collect()

        Xtr = reshape_for_cnn(X_train)
        Xv = reshape_for_cnn(X_val)
        Xte = reshape_for_cnn(X_test)

        cnn = build_cnn(Xtr.shape[1:])

        callback_list = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_auc",
                mode="max",
                patience=8,
                min_delta=1e-4,
                restore_best_weights=True,
                verbose=1,
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                mode="min",
                factor=0.5,
                patience=4,
                min_lr=1e-6,
                verbose=1,
            ),
        ]

        history = cnn.fit(
            Xtr,
            y_train,
            validation_data=(Xv, y_val),
            epochs=100,
            batch_size=512,
            class_weight=class_weight,
            callbacks=callback_list,
            verbose=1,
        )

        cnn_val = cnn.predict(
            Xv,
            batch_size=1024,
            verbose=0,
        ).ravel().astype(np.float32, copy=False)

        cnn_test = cnn.predict(
            Xte,
            batch_size=1024,
            verbose=0,
        ).ravel().astype(np.float32, copy=False)

        cnn_threshold = find_best_threshold(
            y_val,
            cnn_val,
        )

        results.append(
            evaluate(
                y_test,
                cnn_test,
                cnn_threshold,
                f"CNN {feature.title()} — Test",
            )
        )

        cnn.save(
            model_dir / f"cnn_{feature}.keras"
        )

        pd.DataFrame(
            history.history
        ).to_csv(
            model_dir / f"cnn_{feature}_history.csv",
            index=False,
        )

        cnn_models[feature] = cnn

        val_probs[("cnn", feature)] = cnn_val
        test_probs[("cnn", feature)] = cnn_test
        thresholds[("cnn", feature)] = cnn_threshold

        print(
            f"CNN completed in "
            f"{time.perf_counter() - cnn_start:.1f} seconds"
        )
        print(f"CNN threshold: {cnn_threshold:.2f}")

        # ==============================================================
        # Probability ensemble
        # ==============================================================
        setting = find_best_ensemble(
            y_val,
            cnn_val,
            xgb_val,
        )

        ensemble_settings[feature] = setting

        ensemble_test = (
            setting["cnn_weight"] * cnn_test
            + setting["xgb_weight"] * xgb_test
        ).astype(np.float32, copy=False)

        results.append(
            evaluate(
                y_test,
                ensemble_test,
                setting["threshold"],
                f"Ensemble {feature.title()} — Test",
            )
        )

        test_probs[("ensemble", feature)] = ensemble_test

        print(
            "Best ensemble:",
            f"CNN={setting['cnn_weight']:.2f},",
            f"XGB={setting['xgb_weight']:.2f},",
            f"threshold={setting['threshold']:.2f}",
        )

        print(
            f"Feature set completed in "
            f"{time.perf_counter() - feature_start:.1f} seconds"
        )

        # Release temporary reshaped arrays.
        del Xtr, Xv, Xte
        gc.collect()

    # ------------------------------------------------------------------
    # Save results
    # ------------------------------------------------------------------
    results_df = (
        pd.DataFrame(results)
        .sort_values(
            ["f1", "roc_auc"],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    results_df.to_csv(
        model_dir / "final_model_results.csv",
        index=False,
    )

    ensemble_settings_df = pd.DataFrame(
        [
            {
                "feature_set": feature,
                **setting,
            }
            for feature, setting in ensemble_settings.items()
        ]
    )

    ensemble_settings_df.to_csv(
        model_dir / "ensemble_settings.csv",
        index=False,
    )

    xgb_thresholds_df = pd.DataFrame(
        [
            {
                "feature_set": feature,
                "threshold": thresholds[("xgb", feature)],
            }
            for feature in FEATURE_NAMES
        ]
    )

    xgb_thresholds_df.to_csv(
        model_dir / "xgb_thresholds.csv",
        index=False,
    )

    cnn_thresholds_df = pd.DataFrame(
        [
            {
                "feature_set": feature,
                "threshold": thresholds[("cnn", feature)],
            }
            for feature in FEATURE_NAMES
        ]
    )

    cnn_thresholds_df.to_csv(
        model_dir / "cnn_thresholds.csv",
        index=False,
    )

    prediction_data = {
        "actual_label": y_test,
    }

    for architecture in ["xgb", "cnn", "ensemble"]:
        for feature in FEATURE_NAMES:
            key = (architecture, feature)

            if key not in test_probs:
                raise KeyError(
                    f"Missing test probabilities for {key}"
                )

            prediction_data[
                f"{architecture}_{feature}_prob"
            ] = test_probs[key]

    pd.DataFrame(
        prediction_data
    ).to_csv(
        model_dir / "test_probabilities.csv",
        index=False,
    )

    elapsed = time.perf_counter() - experiment_start

    print()
    print(results_df.to_string(index=False))
    print()
    print(
        f"Saved models and results to: "
        f"{model_dir.resolve()}"
    )
    print(
        f"Experiment time: "
        f"{elapsed / 60:.2f} minutes"
    )

    # Release experiment-level data.
    del feature_sets
    gc.collect()

TensorFlow GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Precision policy: <DTypePolicy "mixed_float16">

Experiment: monthly_ff3
Skipping completed experiment: monthly_ff3

Experiment: monthly_ff5
Skipping completed experiment: monthly_ff5

Experiment: daily_ff3
Train rows: 458,325
Validation rows: 98,225
Test rows: 98,225
Positive-class weight: 4.000
Signature  shape: (458325, 21)
Garch      shape: (458325, 12)
Combined   shape: (458325, 33)

--------------------------------------------------------------------------------
Training feature set: signature
--------------------------------------------------------------------------------
XGBoost device: CUDA
XGBoost completed in 0.4 seconds
XGBoost threshold: 0.05


W0000 00:00:1785348917.726207    5498 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1785348917.731316    5498 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9177 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5070, pci bus id: 0000:01:00.0, compute capability: 12.0a


Epoch 1/100


I0000 00:00:1785348920.780303    6684 service.cc:153] XLA service 0x7ae044036420 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1785348920.780337    6684 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5070, Compute Capability 12.0a (Driver: 13.3.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.24.0)
I0000 00:00:1785348920.806914    6684 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1785348921.030312    6684 cuda_dnn.cc:461] Loaded cuDNN version 92400
I0000 00:00:1785348921.075520    6684 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_3302__.29


 56/896 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.5000 - loss: 1.1147

...

540/540 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - auc: 0.5137 - loss: 1.1107 - val_auc: 0.5109 - val_loss: 0.7152 - learning_rate: 0.0010
Epoch 2/100
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.5224 - loss: 1.1080 - val_auc: 0.5104 - val_loss: 0.7192 - learning_rate: 0.0010
Epoch 3/100
540/540 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - auc: 0.5245 - loss: 1.1076 - val_auc: 0.5124 - val_loss: 0.7247 - learning_rate: 0.0010
Epoch 4/100
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.5263 - loss: 1.1073 - val_auc: 0.5106 - val_loss: 0.7328 - learning_rate: 0.0010
Epoch 5/100
533/540 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.5264 - loss: 1.1073
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.5264 - loss: 1.1072 - val_auc: 0.5114 - val_loss: 0.7254 - learning_rate: 0.0010
Epoch 6/100
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.5286 - loss: 1.1069 - val_auc: 0.5112 - val_loss: 0.7272 - learning_rate: 5.0000e-04
Epoc

I0000 00:00:1785349296.456746    6687 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_599872__.29


524/540 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.5101 - loss: 1.1102

I0000 00:00:1785349300.199355    6682 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_599872__.29


540/540 ━━━━━━━━━━━━━━━━━━━━ 8s 9ms/step - auc: 0.5106 - loss: 1.1103 - val_auc: 0.4998 - val_loss: 0.7104 - learning_rate: 0.0010
Epoch 2/100
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.5126 - loss: 1.1091 - val_auc: 0.5016 - val_loss: 0.6947 - learning_rate: 0.0010
Epoch 3/100
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.5137 - loss: 1.1087 - val_auc: 0.4976 - val_loss: 0.6949 - learning_rate: 0.0010
Epoch 4/100
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.5156 - loss: 1.1086 - val_auc: 0.5046 - val_loss: 0.6978 - learning_rate: 0.0010
Epoch 5/100
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.5188 - loss: 1.1082 - val_auc: 0.5022 - val_loss: 0.7025 - learning_rate: 0.0010
Epoch 6/100
540/540 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - auc: 0.5196 - loss: 1.1081
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
540/540 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc: 0.5196 - loss: 1.1081 - val_auc: 0.5047 - val_loss: 0.7057 - learning_rate: 0.0010
Epoch 7/